# LRAA-v13: resumable evaluation of all six completed checkpoints

This notebook performs **no training**. It loads the three parameter-matched
token-mixer and three standard-LoRA checkpoints from Google Drive and runs only
the controls needed to finish the manuscript:

1. normal BoolQ held-out evaluation;
2. adapter-off intervention;
3. the 58-pair human-validated polarity contrast audit.

Each held-out model/seed/condition is saved separately to Drive immediately.
If Colab disconnects, rerun all cells: completed evaluations are skipped. The
final ZIP contains metrics, raw predictions, paired tests, contrast results,
configuration, environment, and hashes. No checkpoint is retrained or modified.


In [ ]:
# FIRST CELL TO RUN: Google Drive permission is requested before any long operation.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected.')


In [ ]:
# Clone the public reproducibility repository automatically. No manual upload is needed.
from pathlib import Path
import shutil, subprocess

REPO_URL = 'https://' + 'github.com/kurbankotan/Adapter-Contribution-Audit.git'
REPO_DIR = Path('/content/Adapter-Contribution-Audit')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

required = [
    REPO_DIR / 'results/v9_reproduction/table_run_results.csv',
    REPO_DIR / 'results/v9_reproduction/evaluation_predictions.csv',
    REPO_DIR / 'results/human_validated_contrast/human_validated_58_contrast_set.jsonl',
    REPO_DIR / 'scripts/score_contrast_set.py',
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Repository is missing required archived artifacts:\n' + '\n'.join(missing))
print('Repository and archived inputs found.')


In [ ]:
# Pinned versions avoid the Colab pandas / Hugging Face dependency conflict seen earlier.
!pip -q uninstall -y gradio >/dev/null 2>&1 || true
!pip -q install --upgrade --no-deps \
  'pandas==2.2.3' 'huggingface-hub==0.36.2' 'transformers==4.57.6' \
  'tokenizers==0.22.1' 'datasets==4.0.0' 'accelerate==1.10.1' 'peft==0.18.1'
!pip -q install 'sentencepiece>=0.2.0' 'protobuf>=5.29,<7' \
  'scikit-learn>=1.5,<1.8' 'seaborn>=0.13,<0.14' 'tabulate>=0.9,<0.10'


In [ ]:
import contextlib, gc, hashlib, importlib.util, json, math, os, platform, random
import shutil, sys, time, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from importlib.metadata import version as package_version

assert package_version('tokenizers') == '0.22.1', package_version('tokenizers')
import transformers
from datasets import load_dataset
from peft import (LoraConfig, TaskType, get_peft_model,
                  get_peft_model_state_dict, set_peft_model_state_dict)
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings('ignore')
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required. In Colab select Runtime > Change runtime type > GPU.')
DEVICE = torch.device('cuda')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA autocast dtype:', AMP_DTYPE)
print('torch / transformers / peft:', torch.__version__, transformers.__version__, __import__('peft').__version__)


In [ ]:
# ----------------------- LOCKED V9 CONFIGURATION -----------------------
DATA_SEED = 2026
SEEDS = [13, 42, 71]
EPOCHS = 3
MAX_LENGTH = 512
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
VAL_TARGET = 1075
SHUFFLE_TRIALS = 3

MIXER_RANK = 496
MIXER_KERNEL = 32
MIXER_DILATION = 16
LORA_RANK = 11
LORA_ALPHA = 22
LORA_DROPOUT = 0.10

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large')
LOCAL_MODEL_DIR = Path('/content/deberta-v3-large')
DRIVE_PROGRESS_DIR = Path('/content/drive/MyDrive/lraa_v13_token_mixer_lora_controls_progress')
RESULT_DIR = Path('/content/lraa_v13_token_mixer_lora_controls_results')
CHECKPOINT_DIR = RESULT_DIR / 'checkpoints'
for p in (DRIVE_PROGRESS_DIR, RESULT_DIR, CHECKPOINT_DIR):
    p.mkdir(parents=True, exist_ok=True)

essential = ['config.json', 'spm.model', 'tokenizer_config.json']
missing = [name for name in essential if not (DRIVE_MODEL_DIR / name).is_file()]
weights = any((DRIVE_MODEL_DIR / name).is_file() for name in ('model.safetensors', 'pytorch_model.bin'))
if missing or not weights:
    raise FileNotFoundError(f'Incomplete model folder: {DRIVE_MODEL_DIR}; missing={missing}; weights={weights}')
shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR, use_fast=True)
SEP_TEXT = tokenizer.sep_token or '[SEP]'

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

RUN_CONFIG = {
    'data_seed': DATA_SEED, 'seeds': SEEDS, 'epochs': EPOCHS,
    'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
    'eval_batch_size': EVAL_BATCH_SIZE, 'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'warmup_ratio': WARMUP_RATIO,
    'shuffle_trials': SHUFFLE_TRIALS,
    'mixer': {'rank': MIXER_RANK, 'kernel': MIXER_KERNEL, 'dilation': MIXER_DILATION},
    'lora': {'rank': LORA_RANK, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
             'target_modules': ['query_proj', 'value_proj']},
    'model_dir': str(DRIVE_MODEL_DIR),
}
(RESULT_DIR / 'run_config.json').write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')


In [ ]:
# Evaluation-only paths. The six final checkpoints already exist on Drive.
CHECKPOINT_SOURCE_DIR = DRIVE_PROGRESS_DIR
EVAL_PROGRESS_DIR = Path('/content/drive/MyDrive/lraa_v13_evaluation_progress')
RESULT_DIR = Path('/content/lraa_v13_evaluation_results')
CHECKPOINT_DIR = CHECKPOINT_SOURCE_DIR
EVAL_PROGRESS_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CHECKPOINTS = [
    *(f'token_mixer_seed{s}.pt' for s in SEEDS),
    *(f'lora_r11_seed{s}.pt' for s in SEEDS),
]
missing = [name for name in EXPECTED_CHECKPOINTS if not (CHECKPOINT_SOURCE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing final checkpoints in {CHECKPOINT_SOURCE_DIR}: {missing}')
print('All six final checkpoints found.')


## Locked passage-grouped data split and candidate serialization


In [ ]:
raw = load_dataset('google/boolq')
train_raw = raw['train'].to_pandas().reset_index(drop=True)
heldout_df = raw['validation'].to_pandas().reset_index(drop=True)

groups = train_raw['passage'].astype(str).to_numpy()
labels = train_raw['answer'].astype(bool).astype(int).to_numpy()
splitter = GroupShuffleSplit(n_splits=80, test_size=VAL_TARGET / len(train_raw), random_state=DATA_SEED)
global_rate = labels.mean()
best = None
for tr_idx, va_idx in splitter.split(np.zeros(len(labels)), labels, groups):
    score = abs(len(va_idx) - VAL_TARGET) / VAL_TARGET + abs(labels[va_idx].mean() - global_rate)
    if best is None or score < best[0]:
        best = (score, tr_idx, va_idx)
_, train_idx, val_idx = best
train_df = train_raw.iloc[train_idx].reset_index(drop=True)
val_df = train_raw.iloc[val_idx].reset_index(drop=True)

assert set(train_df.passage).isdisjoint(set(val_df.passage))
assert (len(train_df), len(val_df)) == (8351, 1076), (len(train_df), len(val_df))
assert len(heldout_df) == 3270

def encode_frame(frame, passages=None):
    passages = frame['passage'].astype(str).tolist() if passages is None else list(map(str, passages))
    questions = frame['question'].astype(str).tolist()
    answers = frame['answer'].astype(bool).astype(int).to_numpy()
    prefix = [f'question: {q} answer:' for q in questions]
    yes_text = [q + ' yes' for q in prefix]
    no_text = [q + ' no' for q in prefix]
    yes = tokenizer(yes_text, passages, truncation='only_second', padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
    no = tokenizer(no_text, passages, truncation='only_second', padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
    return {'yes_ids': yes['input_ids'], 'yes_mask': yes['attention_mask'],
            'no_ids': no['input_ids'], 'no_mask': no['attention_mask'],
            'labels': torch.as_tensor(answers, dtype=torch.long)}

enc_train = encode_frame(train_df)
enc_val = encode_frame(val_df)
enc_heldout = encode_frame(heldout_df)

def derangement(n, seed):
    rng = np.random.default_rng(seed); base = np.arange(n)
    for _ in range(1000):
        perm = rng.permutation(n)
        if np.all(perm != base): return perm
    return np.roll(base, 1)

heldout_passages = heldout_df['passage'].astype(str).to_numpy()
enc_heldout_shuffled = [
    encode_frame(heldout_df, heldout_passages[derangement(len(heldout_df), DATA_SEED + 100 + trial)])
    for trial in range(SHUFFLE_TRIALS)
]
enc_heldout_blank = encode_frame(heldout_df, [''] * len(heldout_df))

data_audit = pd.DataFrame([
    {'split': 'train', 'rows': len(train_df), 'unique_passages': train_df.passage.nunique()},
    {'split': 'internal_validation', 'rows': len(val_df), 'unique_passages': val_df.passage.nunique()},
    {'split': 'heldout', 'rows': len(heldout_df), 'unique_passages': heldout_df.passage.nunique()},
])
data_audit.to_csv(RESULT_DIR / 'data_audit.csv', index=False)
display(data_audit)

# Serialization audit: executable order is tokenizer pair order, identical to V9.
sample_ids = enc_train['yes_ids'][0]
(RESULT_DIR / 'input_serialization_audit.txt').write_text(tokenizer.decode(sample_ids, skip_special_tokens=False), encoding='utf-8')
print((RESULT_DIR / 'input_serialization_audit.txt').read_text(encoding='utf-8')[:700])


In [ ]:
class PairDataset(torch.utils.data.Dataset):
    def __init__(self, encoded, training=False, seed=0, force_swap=False):
        self.encoded = encoded
        self.training = training
        self.seed = int(seed)
        self.force_swap = bool(force_swap)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.encoded['labels'])

    def __getitem__(self, idx):
        if self.training:
            swap = random.Random(self.seed * 1_000_003 + self.epoch * 10_007 + int(idx)).random() < 0.5
        else:
            swap = self.force_swap
        if swap:
            ids = torch.stack([self.encoded['no_ids'][idx], self.encoded['yes_ids'][idx]])
            mask = torch.stack([self.encoded['no_mask'][idx], self.encoded['yes_mask'][idx]])
            semantics = torch.tensor([0, 1], dtype=torch.long)
        else:
            ids = torch.stack([self.encoded['yes_ids'][idx], self.encoded['no_ids'][idx]])
            mask = torch.stack([self.encoded['yes_mask'][idx], self.encoded['no_mask'][idx]])
            semantics = torch.tensor([1, 0], dtype=torch.long)
        gold = int(self.encoded['labels'][idx])
        target = int((semantics == gold).nonzero(as_tuple=False)[0, 0])
        return {'input_ids': ids, 'attention_mask': mask, 'target': target,
                'semantic_labels': semantics, 'gold': gold, 'index': int(idx)}


## Exact matched mixer and standard LoRA models


In [ ]:
class DilatedConvMixerAdapter(nn.Module):
    # Non-attention token mixer with exactly 1,050,625 trainable parameters.
    def __init__(self, hidden=1024, rank=MIXER_RANK, kernel=MIXER_KERNEL, dilation=MIXER_DILATION):
        super().__init__()
        self.depthwise = nn.Conv1d(hidden, hidden, kernel_size=kernel, dilation=dilation,
                                   groups=hidden, bias=False)
        self.down = nn.Linear(hidden, rank, bias=False)
        self.up = nn.Linear(rank, hidden, bias=False)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(0.10)
        self.gate = nn.Parameter(torch.zeros(()))
        self.total_pad = dilation * (kernel - 1)

    def forward(self, hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        x = hidden * mask
        left = self.total_pad // 2
        right = self.total_pad - left
        mixed = self.depthwise(F.pad(x.transpose(1, 2), (left, right))).transpose(1, 2)
        mixed = mixed * mask
        update = self.up(F.gelu(self.down(mixed)))
        return self.norm(hidden + torch.tanh(self.gate) * self.dropout(update))


class MixerPairScorer(nn.Module):
    def __init__(self, hidden=1024):
        super().__init__()
        self.adapter = DilatedConvMixerAdapter(hidden)
        self.dropout = nn.Dropout(0.15)
        self.scorer = nn.Linear(hidden, 1)

    def score_hidden(self, hidden, mask, adapter_enabled=True):
        if adapter_enabled:
            hidden = self.adapter(hidden, mask)
        return self.scorer(self.dropout(hidden[:, 0])).squeeze(-1)


class LoraPairScorer(nn.Module):
    def __init__(self):
        super().__init__()
        base = AutoModel.from_pretrained(LOCAL_MODEL_DIR)
        if hasattr(base.config, 'use_cache'):
            base.config.use_cache = False
        for p in base.parameters():
            p.requires_grad = False
        try:
            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
            base.enable_input_require_grads()
        except Exception as exc:
            print('Gradient-checkpointing setup warning:', exc)
        cfg = LoraConfig(
            r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            bias='none', target_modules=['query_proj', 'value_proj'],
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        self.encoder = get_peft_model(base, cfg)
        self.dropout = nn.Dropout(0.15)
        self.scorer = nn.Linear(base.config.hidden_size, 1)

    def forward(self, ids, mask):
        hidden = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        return self.scorer(self.dropout(hidden[:, 0])).squeeze(-1)


def trainable_count(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

mixer_probe = MixerPairScorer()
MIXER_ADAPTER_PARAMS = trainable_count(mixer_probe.adapter)
MIXER_SCORER_PARAMS = trainable_count(mixer_probe.scorer)
assert MIXER_ADAPTER_PARAMS == 1_050_625, MIXER_ADAPTER_PARAMS
del mixer_probe

lora_probe = LoraPairScorer()
LORA_ADAPTER_PARAMS = sum(p.numel() for p in lora_probe.encoder.parameters() if p.requires_grad)
LORA_SCORER_PARAMS = trainable_count(lora_probe.scorer)
target_counts = {
    suffix: sum(1 for n, _ in lora_probe.encoder.named_modules() if n.endswith(suffix))
    for suffix in ('query_proj', 'value_proj')
}
assert target_counts == {'query_proj': 24, 'value_proj': 24}, target_counts
assert LORA_ADAPTER_PARAMS == 1_081_344, LORA_ADAPTER_PARAMS
del lora_probe; gc.collect(); torch.cuda.empty_cache()

parameter_counts = pd.DataFrame([
    {'model': 'lraa_archived', 'adapter_params': 1_050_625, 'scorer_params': 1_025, 'total_trainable': 1_051_650},
    {'model': 'token_mixer', 'adapter_params': MIXER_ADAPTER_PARAMS, 'scorer_params': MIXER_SCORER_PARAMS,
     'total_trainable': MIXER_ADAPTER_PARAMS + MIXER_SCORER_PARAMS},
    {'model': 'lora_r11', 'adapter_params': LORA_ADAPTER_PARAMS, 'scorer_params': LORA_SCORER_PARAMS,
     'total_trainable': LORA_ADAPTER_PARAMS + LORA_SCORER_PARAMS},
])
parameter_counts['pct_vs_lraa'] = 100 * (parameter_counts.total_trainable / 1_051_650 - 1)
parameter_counts.to_csv(RESULT_DIR / 'parameter_counts.csv', index=False)
display(parameter_counts)


## Training and held-out evaluation


In [ ]:
def semantic_metrics(gold, pred, probs=None, loss=None):
    out = {
        'accuracy': accuracy_score(gold, pred),
        'balanced_accuracy': balanced_accuracy_score(gold, pred),
        'macro_f1': f1_score(gold, pred, average='macro', zero_division=0),
        'predicted_yes_rate': float(np.mean(pred)),
        'true_yes_rate': float(np.mean(gold)),
    }
    if probs is not None:
        out['log_loss'] = log_loss(gold, probs, labels=[0, 1])
    if loss is not None:
        out['loss'] = float(loss)
    return out


def loaders_for(encoded, seed, training=False, swap=False):
    ds = PairDataset(encoded, training=training, seed=seed, force_swap=swap)
    gen = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH_SIZE if training else EVAL_BATCH_SIZE,
                      shuffle=training, generator=gen if training else None,
                      num_workers=2, pin_memory=True, persistent_workers=False)


@torch.no_grad()
def encode_batch(encoder, ids, mask):
    with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
        return encoder(input_ids=ids, attention_mask=mask).last_hidden_state


def flatten_pair(batch):
    ids = batch['input_ids'].to(DEVICE, non_blocking=True)
    mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
    bsz, nopt, seqlen = ids.shape
    return (ids.view(bsz * nopt, seqlen), mask.view(bsz * nopt, seqlen),
            batch['target'].to(DEVICE), batch['gold'].to(DEVICE), bsz)


@torch.no_grad()
def eval_mixer(encoder, model, encoded, seed, adapter_enabled=True, swap=False):
    model.eval(); rows = []; loss_sum = 0.0
    for batch in loaders_for(encoded, seed, training=False, swap=swap):
        ids, mask, target, gold_semantic, bsz = flatten_pair(batch)
        hidden = encode_batch(encoder, ids, mask)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model.score_hidden(hidden, mask, adapter_enabled=adapter_enabled).view(bsz, 2)
            loss = F.cross_entropy(logits.float(), target)
        option_probs = torch.softmax(logits.float(), dim=-1).cpu()
        yes_pos = (batch['semantic_labels'] == 1).long().argmax(dim=1)
        prob1 = option_probs.gather(1, yes_pos.unsqueeze(1)).squeeze(1)
        pred_pos = logits.argmax(-1)
        semantic = batch['semantic_labels'].gather(1, pred_pos.cpu().unsqueeze(1)).squeeze(1)
        rows.extend(zip(batch['index'].tolist(), gold_semantic.cpu().tolist(), semantic.tolist(), prob1.cpu().tolist()))
        loss_sum += loss.item() * bsz
    rows.sort(key=lambda x: x[0])
    idx, gold, pred, prob = map(np.asarray, zip(*rows))
    return semantic_metrics(gold, pred, prob, loss_sum / len(rows)), pd.DataFrame({'example_index': idx, 'gold': gold, 'prediction': pred, 'prob_yes': prob})


@torch.no_grad()
def eval_lora(model, encoded, seed, adapter_enabled=True, swap=False):
    model.eval(); rows = []; loss_sum = 0.0
    cm = contextlib.nullcontext() if adapter_enabled else model.encoder.disable_adapter()
    with cm:
        for batch in loaders_for(encoded, seed, training=False, swap=swap):
            ids, mask, target, gold_semantic, bsz = flatten_pair(batch)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(ids, mask).view(bsz, 2)
                loss = F.cross_entropy(logits.float(), target)
            option_probs = torch.softmax(logits.float(), dim=-1).cpu()
            yes_pos = (batch['semantic_labels'] == 1).long().argmax(dim=1)
            prob1 = option_probs.gather(1, yes_pos.unsqueeze(1)).squeeze(1)
            pred_pos = logits.argmax(-1)
            semantic = batch['semantic_labels'].gather(1, pred_pos.cpu().unsqueeze(1)).squeeze(1)
            rows.extend(zip(batch['index'].tolist(), gold_semantic.cpu().tolist(), semantic.tolist(), prob1.cpu().tolist()))
            loss_sum += loss.item() * bsz
    rows.sort(key=lambda x: x[0])
    idx, gold, pred, prob = map(np.asarray, zip(*rows))
    return semantic_metrics(gold, pred, prob, loss_sum / len(rows)), pd.DataFrame({'example_index': idx, 'gold': gold, 'prediction': pred, 'prob_yes': prob})


## Resumable held-out and adapter-off evaluation


In [ ]:
def atomic_write_json(obj, destination):
    local = Path('/content') / destination.name
    temp = Path(str(destination) + '.tmp')
    local.write_text(json.dumps(obj, indent=2), encoding='utf-8')
    shutil.copy2(local, temp); os.replace(temp, destination)


def atomic_write_csv(frame, destination):
    local = Path('/content') / destination.name
    temp = Path(str(destination) + '.tmp')
    frame.to_csv(local, index=False)
    shutil.copy2(local, temp); os.replace(temp, destination)


def progress_paths(mode, seed, condition):
    stem = f'{mode}_seed{seed}_{condition}'
    return EVAL_PROGRESS_DIR / f'{stem}_metrics.json', EVAL_PROGRESS_DIR / f'{stem}_predictions.csv'


def saved_or_evaluate(mode, seed, condition, evaluator):
    metrics_path, pred_path = progress_paths(mode, seed, condition)
    if metrics_path.is_file() and pred_path.is_file():
        print('Skipped completed evaluation:', metrics_path.stem)
        return json.loads(metrics_path.read_text(encoding='utf-8')), pd.read_csv(pred_path)
    metrics, predictions = evaluator()
    atomic_write_json(metrics, metrics_path)
    atomic_write_csv(predictions, pred_path)
    print('Saved evaluation:', metrics_path.stem)
    return metrics, predictions


run_rows, prediction_frames = [], []

# Token mixer: reuse one frozen encoder for all three checkpoints.
frozen_encoder = AutoModel.from_pretrained(LOCAL_MODEL_DIR).to(DEVICE).eval()
for p in frozen_encoder.parameters(): p.requires_grad = False
for seed in SEEDS:
    payload = torch.load(CHECKPOINT_SOURCE_DIR / f'token_mixer_seed{seed}.pt',
                         map_location='cpu', weights_only=False)
    model = MixerPairScorer().to(DEVICE)
    model.load_state_dict(payload['model_state_dict']); model.eval()
    normal, normal_pred = saved_or_evaluate(
        'token_mixer', seed, 'normal',
        lambda: eval_mixer(frozen_encoder, model, enc_heldout, seed, adapter_enabled=True))
    off, off_pred = saved_or_evaluate(
        'token_mixer', seed, 'adapter_off',
        lambda: eval_mixer(frozen_encoder, model, enc_heldout, seed, adapter_enabled=False))
    run_rows.append({'model':'token_mixer','seed':seed,'best_epoch':int(payload['best_epoch']),
                     **{f'normal_{k}':v for k,v in normal.items()},
                     **{f'adapter_off_{k}':v for k,v in off.items()},
                     'adapter_gain_accuracy':normal['accuracy']-off['accuracy'],
                     'adapter_gain_balanced_accuracy':normal['balanced_accuracy']-off['balanced_accuracy'],
                     'adapter_gain_macro_f1':normal['macro_f1']-off['macro_f1']})
    normal_pred['model']='token_mixer'; normal_pred['seed']=seed; prediction_frames.append(normal_pred)
    del model; gc.collect(); torch.cuda.empty_cache()
del frozen_encoder; gc.collect(); torch.cuda.empty_cache()

# Standard LoRA: load and release one adapted encoder at a time.
for seed in SEEDS:
    payload = torch.load(CHECKPOINT_SOURCE_DIR / f'lora_r11_seed{seed}.pt',
                         map_location='cpu', weights_only=False)
    model = LoraPairScorer().to(DEVICE)
    set_peft_model_state_dict(model.encoder, payload['adapter_state_dict'])
    model.scorer.load_state_dict(payload['scorer_state_dict']); model.eval()
    normal, normal_pred = saved_or_evaluate(
        'lora_r11', seed, 'normal',
        lambda: eval_lora(model, enc_heldout, seed, adapter_enabled=True))
    off, off_pred = saved_or_evaluate(
        'lora_r11', seed, 'adapter_off',
        lambda: eval_lora(model, enc_heldout, seed, adapter_enabled=False))
    run_rows.append({'model':'lora_r11','seed':seed,'best_epoch':int(payload['best_epoch']),
                     **{f'normal_{k}':v for k,v in normal.items()},
                     **{f'adapter_off_{k}':v for k,v in off.items()},
                     'adapter_gain_accuracy':normal['accuracy']-off['accuracy'],
                     'adapter_gain_balanced_accuracy':normal['balanced_accuracy']-off['balanced_accuracy'],
                     'adapter_gain_macro_f1':normal['macro_f1']-off['macro_f1']})
    normal_pred['model']='lora_r11'; normal_pred['seed']=seed; prediction_frames.append(normal_pred)
    del model; gc.collect(); torch.cuda.empty_cache()

run_df = pd.DataFrame(run_rows)
pred_df = pd.concat(prediction_frames, ignore_index=True)
run_df.to_csv(RESULT_DIR / 'control_run_results.csv', index=False)
pred_df.to_csv(RESULT_DIR / 'heldout_predictions.csv', index=False)
display(run_df[['model','seed','best_epoch','normal_accuracy','normal_balanced_accuracy',
                'normal_macro_f1','adapter_off_balanced_accuracy','adapter_gain_balanced_accuracy']])


## Archived-LRAA comparison and prespecified topology decision


In [ ]:
arch_runs = pd.read_csv(REPO_DIR / 'results/v9_reproduction/table_run_results.csv')
arch_pred = pd.read_csv(REPO_DIR / 'results/v9_reproduction/evaluation_predictions.csv')
lraa_runs = arch_runs[arch_runs.model.astype(str).str.contains('lcca|lraa', case=False, regex=True)].copy()
lraa_pred = arch_pred[arch_pred.model.astype(str).str.contains('lcca|lraa', case=False, regex=True)].copy()
assert set(lraa_runs.seed.astype(int)) == set(SEEDS)

def t95(values):
    values = np.asarray(values, dtype=float); mean = values.mean()
    half = 4.3026527299 * values.std(ddof=1) / math.sqrt(len(values))
    return mean, mean-half, mean+half

def mcnemar_exact(gold, a, b):
    ac = np.asarray(a) == np.asarray(gold); bc = np.asarray(b) == np.asarray(gold)
    b_only = int(np.sum(ac & ~bc)); c_only = int(np.sum(~ac & bc)); n = b_only + c_only
    return {'a_only_correct': b_only, 'b_only_correct': c_only,
            'p_exact': 1.0 if n == 0 else float(binomtest(min(b_only,c_only), n, 0.5).pvalue)}

comparisons = []; mcnemar_rows = []
for control in ('token_mixer', 'lora_r11'):
    diffs = []
    for seed in SEEDS:
        lr = lraa_runs[lraa_runs.seed.astype(int) == seed].iloc[0]
        cr = run_df[(run_df.model == control) & (run_df.seed == seed)].iloc[0]
        diff = float(lr.normal_balanced_accuracy - cr.normal_balanced_accuracy)
        diffs.append(diff)
        ap = lraa_pred[lraa_pred.seed.astype(int) == seed].sort_values('example_index')
        bp = pred_df[(pred_df.model == control) & (pred_df.seed == seed)].sort_values('example_index')
        audit = mcnemar_exact(ap.gold.astype(int), ap.prediction.astype(int), bp.prediction.astype(int))
        mcnemar_rows.append({'comparison': f'lraa_minus_{control}', 'seed': seed, **audit})
    mean, low, high = t95(diffs)
    comparisons.append({'comparison': f'lraa_minus_{control}', 'metric': 'balanced_accuracy',
                        'seed_differences': diffs, 'mean_difference': mean, 'ci95_low': low, 'ci95_high': high,
                        'all_seeds_lraa_higher': all(x > 0 for x in diffs)})

mix = next(x for x in comparisons if x['comparison'] == 'lraa_minus_token_mixer')
topology_pass = bool(mix['mean_difference'] >= 0.005 and mix['ci95_low'] > 0 and mix['all_seeds_lraa_higher'])
decision = {
    'prespecified_question': 'Does LRAA exceed the parameter-matched non-attention token mixer?',
    'rule': 'PASS only if mean balanced-accuracy advantage is at least 0.5 percentage points, LRAA wins all three seeds, and the seed-level 95% t interval excludes zero.',
    'decision': 'PASS' if topology_pass else 'FAIL',
    'interpretation_if_fail': 'The result does not isolate attention topology from generic token mixing.',
    'comparisons': comparisons,
}
(RESULT_DIR / 'control_comparison.json').write_text(json.dumps(decision, indent=2), encoding='utf-8')
pd.DataFrame(mcnemar_rows).to_csv(RESULT_DIR / 'heldout_mcnemar.csv', index=False)

summary_rows = []
for model in ('token_mixer','lora_r11'):
    part = run_df[run_df.model == model]
    summary_rows.append({'model': model, 'accuracy_mean': part.normal_accuracy.mean(), 'accuracy_sd': part.normal_accuracy.std(ddof=1),
                         'balanced_accuracy_mean': part.normal_balanced_accuracy.mean(), 'balanced_accuracy_sd': part.normal_balanced_accuracy.std(ddof=1),
                         'macro_f1_mean': part.normal_macro_f1.mean(), 'macro_f1_sd': part.normal_macro_f1.std(ddof=1)})
summary_rows.append({'model': 'lraa_archived', 'accuracy_mean': lraa_runs.normal_accuracy.mean(), 'accuracy_sd': lraa_runs.normal_accuracy.std(ddof=1),
                     'balanced_accuracy_mean': lraa_runs.normal_balanced_accuracy.mean(), 'balanced_accuracy_sd': lraa_runs.normal_balanced_accuracy.std(ddof=1),
                     'macro_f1_mean': lraa_runs.normal_macro_f1.mean(), 'macro_f1_sd': lraa_runs.normal_macro_f1.std(ddof=1)})
summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULT_DIR / 'control_summary.csv', index=False)
table = summary.to_markdown(index=False, floatfmt='.4f')
(RESULT_DIR / 'control_summary.md').write_text(table + '\n', encoding='utf-8')
print(table)
print('\nPrespecified token-mixer decision:', decision['decision'])
display(pd.DataFrame(comparisons))


## Resumable human-validated polarity contrast evaluation


In [ ]:
score_path = REPO_DIR / 'scripts/score_contrast_set.py'
spec = importlib.util.spec_from_file_location('score_contrast_set', score_path)
score_mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(score_mod)
contrast_path = REPO_DIR / 'results/human_validated_contrast/human_validated_58_contrast_set.jsonl'
contrast_rows = score_mod.load(str(contrast_path))
assert len(contrast_rows) == 58

def encode_candidate(passage, question, option):
    first = f'question: {question} answer: {option}'
    batch = tokenizer(first, passage, truncation='only_second', padding='max_length',
                      max_length=MAX_LENGTH, return_tensors='pt')
    return batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE)

def contrast_progress_paths(mode, seed):
    stem = f'{mode}_seed{seed}_contrast58'
    return EVAL_PROGRESS_DIR / f'{stem}_metrics.json', EVAL_PROGRESS_DIR / f'{stem}_predictions.csv'

def saved_or_score_contrast(mode, seed, score_fn):
    metrics_path, pred_path = contrast_progress_paths(mode, seed)
    if metrics_path.is_file() and pred_path.is_file():
        print('Skipped completed contrast:', metrics_path.stem)
        return json.loads(metrics_path.read_text(encoding='utf-8')), pd.read_csv(pred_path).to_dict('records')
    predictions = score_mod.score_arms(contrast_rows, score_fn, mode, seed)
    metrics = score_mod.compute_metrics(contrast_rows, predictions)
    atomic_write_json(metrics, metrics_path)
    atomic_write_csv(pd.DataFrame(predictions), pred_path)
    print('Saved contrast:', metrics_path.stem)
    return metrics, predictions

all_contrast_predictions, all_contrast_metrics = [], []

frozen_encoder = AutoModel.from_pretrained(LOCAL_MODEL_DIR).to(DEVICE).eval()
for p in frozen_encoder.parameters(): p.requires_grad = False
for seed in SEEDS:
    payload = torch.load(CHECKPOINT_SOURCE_DIR / f'token_mixer_seed{seed}.pt',
                         map_location='cpu', weights_only=False)
    model = MixerPairScorer().to(DEVICE); model.load_state_dict(payload['model_state_dict']); model.eval()
    @torch.no_grad()
    def score_fn(passage, question, option, _model=model):
        ids, mask = encode_candidate(passage, question, option)
        hidden = encode_batch(frozen_encoder, ids, mask)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            return float(_model.score_hidden(hidden, mask, True).item())
    metrics, predictions = saved_or_score_contrast('token_mixer', seed, score_fn)
    all_contrast_metrics.append({'model':'token_mixer','seed':seed,**metrics})
    all_contrast_predictions.extend(predictions)
    del model; gc.collect(); torch.cuda.empty_cache()
del frozen_encoder; gc.collect(); torch.cuda.empty_cache()

for seed in SEEDS:
    payload = torch.load(CHECKPOINT_SOURCE_DIR / f'lora_r11_seed{seed}.pt',
                         map_location='cpu', weights_only=False)
    model = LoraPairScorer().to(DEVICE)
    set_peft_model_state_dict(model.encoder, payload['adapter_state_dict'])
    model.scorer.load_state_dict(payload['scorer_state_dict']); model.eval()
    @torch.no_grad()
    def score_fn(passage, question, option, _model=model):
        ids, mask = encode_candidate(passage, question, option)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            return float(_model(ids, mask).item())
    metrics, predictions = saved_or_score_contrast('lora_r11', seed, score_fn)
    all_contrast_metrics.append({'model':'lora_r11','seed':seed,**metrics})
    all_contrast_predictions.extend(predictions)
    del model; gc.collect(); torch.cuda.empty_cache()

pd.DataFrame(all_contrast_predictions).to_csv(RESULT_DIR / 'contrast_predictions_validated58.csv', index=False)
(RESULT_DIR / 'contrast_metrics_validated58.json').write_text(
    json.dumps(all_contrast_metrics, indent=2), encoding='utf-8')

rows = []
for mode in ('token_mixer','lora_r11'):
    part = [x for x in all_contrast_metrics if x['model'] == mode]
    rows.append({
        'model':mode,
        'contrast_consistency':float(np.mean([x['contrast_consistency'] for x in part])),
        'A_accuracy':float(np.mean([x['accuracy']['a'] for x in part])),
        'B_accuracy':float(np.mean([x['accuracy']['b'] for x in part])),
        'flip_rate':float(np.mean([x['flip_rate'] for x in part])),
        'edit_cost':float(np.mean([x['edit_cost'] for x in part])),
    })
contrast_summary = pd.DataFrame(rows)
contrast_summary.to_csv(RESULT_DIR / 'contrast_summary_validated58.csv', index=False)
table = contrast_summary.to_markdown(index=False, floatfmt='.4f')
(RESULT_DIR / 'contrast_table_validated58.md').write_text(table+'\n', encoding='utf-8')
print(table)


## Integrity record, ZIP, and automatic download


In [ ]:
environment = {
    'python':sys.version, 'platform':platform.platform(), 'torch':torch.__version__,
    'transformers':transformers.__version__, 'peft':__import__('peft').__version__,
    'cuda':torch.version.cuda, 'gpu':torch.cuda.get_device_name(0), 'amp_dtype':str(AMP_DTYPE),
}
(RESULT_DIR / 'run_environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
(RESULT_DIR / 'run_config.json').write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')

checkpoint_manifest = []
for name in EXPECTED_CHECKPOINTS:
    path = CHECKPOINT_SOURCE_DIR / name
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checkpoint_manifest.append({'file':name,'bytes':path.stat().st_size,'sha256':digest})
pd.DataFrame(checkpoint_manifest).to_csv(RESULT_DIR / 'checkpoint_manifest_sha256.csv', index=False)

manifest = []
for path in sorted(RESULT_DIR.rglob('*')):
    if path.is_file():
        manifest.append({'file':str(path.relative_to(RESULT_DIR)),'bytes':path.stat().st_size,
                         'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
pd.DataFrame(manifest).to_csv(RESULT_DIR / 'result_file_manifest_sha256.csv', index=False)

zip_path = Path('/content/lraa_v13_all_checkpoint_evaluation_results.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(RESULT_DIR.rglob('*')):
        if path.is_file(): zf.write(path, path.relative_to(RESULT_DIR.parent))
print('Created:', zip_path, f'({zip_path.stat().st_size/1e6:.1f} MB)')
from google.colab import files
files.download(str(zip_path))
print('Download requested. Runtime was NOT disconnected or deleted.')
